# VCuboidFIT — chạy trực tiếp từ ZIP trên Google Drive
Chỉ cần chạy **4 ô** theo thứ tự. Dữ liệu ở `MyDrive/datatest/`, kết quả ở `MyDrive/vcf_results/`.
Dùng runtime CPU mặc định; không cần tài khoản dữ liệu, Docker, token hay kết nối máy cá nhân.
Các ZIP phải là các phần của **cùng một bộ nuScenes**, chứa bảng JSON `v1.0-*` và LiDAR `.bin`.
Nếu folder của bạn nằm chỗ khác, chỉ sửa `DATA_FOLDER` trong ô 1.

## 1. Kết nối Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys
import tempfile

drive.mount('/content/drive')
DATA_FOLDER = Path('/content/drive/MyDrive/datatest')
RESULT_FOLDER = Path('/content/drive/MyDrive/vcf_results')
REPO = Path('/content/VCuboidFit')
if not DATA_FOLDER.is_dir():
    raise FileNotFoundError(f'Không thấy {DATA_FOLDER}. Sửa DATA_FOLDER thành đường dẫn folder ZIP của bạn.')
print('Folder dữ liệu:', DATA_FOLDER)


## 2. Cài phần CPU (không build GPU)

In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Nguyendev037/VCuboidFit.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                str(REPO / 'model/worker')], check=True)
sys.path.insert(0, str(REPO / 'model/worker'))
os.environ['PYTHONPATH'] = str(REPO / 'model/worker')
import c4.lidar
print('SETUP OK — sẵn sàng đọc ZIP và chạy CPU')


## 3. Đọc tất cả ZIP trong datatest và kiểm tra dữ liệu
Tự giải nén vào ổ tạm Colab. Giữ nguyên các ZIP trên Drive; không cần giải nén tay.

In [ ]:
import filecmp
from datetime import datetime
from service.archives import DatasetError, _safe_zip_extract, merge_into
from c4.data.nusc import NuscTables
from c4.lidar.index import build_lidar_index

archives = sorted(p for p in DATA_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == '.zip')
if not archives:
    raise FileNotFoundError(f'Không có file .zip trong {DATA_FOLDER}.')
RUN = Path(tempfile.mkdtemp(prefix='vcf-', dir='/content'))
NUSC = RUN / 'data'
JOB = RUN / 'job'
RUN_NAME = datetime.now().strftime('%Y%m%d-%H%M%S') + '-' + RUN.name
for i, archive in enumerate(archives, 1):
    print(f'[{i}/{len(archives)}] Giải nén {archive.name}', flush=True)
    with tempfile.TemporaryDirectory(dir=RUN) as raw_dir:
        raw = Path(raw_dir)
        _safe_zip_extract(archive, raw)
        candidates = [raw, *(p for p in raw.rglob('*') if p.is_dir())]
        roots = [p for p in candidates if (p / 'samples').is_dir()
                 or any(v.is_dir() for v in p.glob('v1.0-*'))
                 or (p / 'sweeps').is_dir()]
        if not roots:
            raise DatasetError(f'{archive.name}: không thấy samples/, sweeps/ hay v1.0-*. Cần ZIP dữ liệu nuScenes.')
        for root in roots:
            for f in root.rglob('*'):
                if f.is_file():
                    dest = NUSC / f.relative_to(root)
                    if dest.is_file() and not filecmp.cmp(f, dest, shallow=False):
                        raise DatasetError(f'{archive.name}: file trùng nhưng nội dung khác: {f.relative_to(root)}. Chỉ để ZIP của cùng một dataset trong folder.')
        merge_into(roots, NUSC)
tables = NuscTables.load(NUSC)
index = build_lidar_index(tables, NUSC)
if index.empty:
    raise DatasetError('Không có keyframe LIDAR_TOP. Cần ZIP chứa samples/LIDAR_TOP/*.bin và bảng JSON tương ứng.')
print(f'DATA OK — {len(archives)} ZIP, {index.scene_token.nunique()} scene, {len(index)} keyframe LiDAR')


## 4. Chạy chọn 5% và lưu kết quả về Drive
Chạy trên toàn bộ keyframe của bộ test. Không cần nhãn; không tune hay chấm tập P.

In [ ]:
import shutil
import numpy as np
import pandas as pd
from c4.contracts import write_table
from c4.lidar import load_lidar_config
from c4.lidar.extract import extract_all, embed_split
from c4.lidar.params import LidarParams
from c4.lidar.pipeline import run_selection_lidar

cfg = load_lidar_config()
JOB.mkdir(parents=True, exist_ok=True)
lid = JOB / 'lidar'
lid.mkdir(exist_ok=True)
write_table(index, lid / 'index.parquet', 'lidar_index')
raw, filt = extract_all(index, NUSC, lid, cfg, n_jobs=2)
keep = filt['keep'].to_numpy(bool)
if keep.sum() < 2:
    raise ValueError('Cần ít nhất 2 frame LiDAR hợp lệ. Kiểm tra file .bin và số điểm mỗi frame.')
np.save(lid / 'z0.npy', embed_split(raw, keep, cfg))

def save_selection(tier):
    result = run_selection_lidar(JOB, LidarParams(tier=tier))
    source = JOB / 'out/selections' / result['selectionId']
    target = RESULT_FOLDER / RUN_NAME / f'tier{tier}'
    target.mkdir(parents=True, exist_ok=True)
    for name in ('result.json', 'selected.csv', 'scores.parquet'):
        shutil.copy2(source / name, target / name)
    all_selected = pd.read_csv(target / 'selected.csv')
    selected = all_selected[(all_selected['method'] == 'hybrid')
                            & (all_selected['rank'] <= result['budgetB'])]
    selected.to_csv(target / 'selected_5pct.csv', index=False)
    bundle = shutil.make_archive(str(target / 'results'), 'zip', root_dir=source)
    print(f'XONG — chọn {len(selected)}/{result["poolSize"]} frame. Kết quả: {target}')
    print('File ZIP kết quả:', bundle)
    if result['warnings']:
        print('Lưu ý:', '; '.join(result['warnings']))
    print(selected.head(20).to_string(index=False))
    return result

result = save_selection(0)


## Tùy chọn: Tầng 1 GPU
**Đã có kết quả sau ô 4.** Chỉ chạy phần này nếu muốn train PointPillars.
Đổi Runtime → Change runtime type → T4 GPU, rồi chạy lại ô 1–4 nếu Colab khởi động lại.
Build GPU mất thời gian và có thể phụ thuộc phiên bản runtime. Dữ liệu phải có scene seed S
(theo cấu hình split nuScenes chuẩn); dữ liệu test tùy ý không đảm bảo train được.

In [ ]:
RUN_GPU = False  # @param {type:"boolean"}
if RUN_GPU:
    from c4.lidar.splits import assign_splits
    seed_index = assign_splits(index, cfg)
    if not (seed_index['split'] == 'S').any():
        raise ValueError('Dataset không có scene seed S; dùng kết quả CPU ở ô 4 hoặc chuẩn bị dữ liệu seed nuScenes.')
    subprocess.run(['nvidia-smi'], check=True)
    os.environ['VCF_HOME'] = '/content/vcf_gpu'
    subprocess.run(['bash', str(REPO / 'model/scripts/colab_setup.sh')], check=True)
    os.environ['PCDET_ROOT'] = '/content/vcf_gpu/OpenPCDet'
    os.environ['PYTHONPATH'] = str(REPO / 'model/worker') + ':' + os.environ['PCDET_ROOT']
    EXP = RUN / 'exp'
    EXP.mkdir(exist_ok=True)
    write_table(seed_index, EXP / 'index.parquet', 'lidar_index')
    subprocess.run(['bash', str(REPO / 'model/scripts/tier1.sh'), str(NUSC), str(EXP),
                    '--no-docker', '--sweeps', '1', '--epochs', '20', '--batch', '2'], check=True)
    shutil.copytree(EXP / 't1', JOB / 't1', dirs_exist_ok=True)
    result_gpu = save_selection(1)
else:
    print('Đã có kết quả CPU. GPU đang tắt; chỉ bật RUN_GPU nếu muốn train PointPillars.')
